
# NY–NJ DEM 洪水易发性因子提取

这个 Notebook 基于你的最终 DEM：

```text
D:\python\DEM_work\data\Features_resolution_30_m\DEM.tif
```

特点：

- 以 `rasterio` 为主进行 DEM 分块读取、写出和重采样。
- 对大栅格友好：地形因子全部分块计算，避免一次性读入整幅 DEM。
- 对距离因子友好：`DTRiver / DTRoad / DTDrainage` 默认使用 GDAL Proximity 写盘计算，避免 `scipy.distance_transform_edt` 一次性占用大量内存。
- 支持可选输入：AP 降雨、CN、河流、道路、排水线。
- 自动输出压缩 GeoTIFF，并可建立金字塔，方便 QGIS / ArcGIS 显示。

建议先运行到 `DEM 信息检查`，确认 DEM 已经是投影坐标系、单位为米。若 DEM 是经纬度坐标，请先投影到 `EPSG:5070` 或适合研究区的米制投影。


In [63]:
# -*- coding: utf-8 -*-
# =========================================================
# 0. 环境与导入
# 注意：若你之前在同一个 Notebook 内已经 import 过 rasterio/geopandas，
# 修改 PROJ/GDAL 环境变量后建议 Restart Kernel 再从头运行。
# =========================================================

import os
import sys
from pathlib import Path
from typing import Optional, List, Tuple, Dict

# 强制指定当前 conda 环境中的 PROJ / GDAL 数据目录
CONDA_ENV_DIR = Path(os.environ.get("CONDA_PREFIX", Path(sys.executable).resolve().parent))
PROJ_DIR = CONDA_ENV_DIR / "Library" / "share" / "proj"
GDAL_DIR = CONDA_ENV_DIR / "Library" / "share" / "gdal"

os.environ["PROJ_DATA"] = str(PROJ_DIR)
os.environ["PROJ_LIB"] = str(PROJ_DIR)
os.environ["GDAL_DATA"] = str(GDAL_DIR)

print("Python:", sys.executable)
print("CONDA_ENV_DIR:", CONDA_ENV_DIR)
print("PROJ_DIR:", PROJ_DIR, "exists=", (PROJ_DIR / "proj.db").exists())
print("GDAL_DIR:", GDAL_DIR, "exists=", GDAL_DIR.exists())

import warnings
warnings.filterwarnings("ignore", category=RuntimeWarning)

import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio

from rasterio.enums import Resampling
from rasterio.features import rasterize
from rasterio.windows import Window
from rasterio.warp import reproject
from rasterio.transform import Affine

from scipy.ndimage import uniform_filter, maximum_filter, minimum_filter
from scipy.spatial import cKDTree

from shapely.geometry import Point
from tqdm.auto import tqdm

# GDAL 只用于大规模矢量栅格化和距离变换，避免全图 distance_transform_edt 占用巨量内存
try:
    from osgeo import gdal, ogr, osr
    GDAL_AVAILABLE = True
    gdal.UseExceptions()
except Exception as e:
    GDAL_AVAILABLE = False
    print("[警告] osgeo.gdal 不可用，DTRiver/DTRoad/DTDrainage 的距离计算会被跳过。")
    print("建议安装：conda install -c conda-forge gdal")
    print("错误：", e)


Python: d:\miniconda\envs\dem_work\python.exe
CONDA_ENV_DIR: D:\miniconda\envs\dem_work
PROJ_DIR: D:\miniconda\envs\dem_work\Library\share\proj exists= True
GDAL_DIR: D:\miniconda\envs\dem_work\Library\share\gdal exists= True


In [64]:
# =========================================================
# 1. 用户配置区：主要改这里
# =========================================================

# 你的最终 DEM
DEM_PATH = Path(r"D:\python\DEM_work\data\Features_resolution_30_m\DEM.tif")

# 输出目录
OUT_DIR = Path(r"D:\python\DEM_work\data\Features_resolution_30_m")

# 输出 NoData 和数据类型
NODATA = -9999.0
OUT_DTYPE = "float32"

# 分块大小：越大越快，但越占内存
# 30m DEM，建议 1024~2048；内存较小就用 1024
BLOCK_SIZE_TERRAIN = 2048
BLOCK_SIZE_IDW = 512

# GDAL 多线程和缓存
NUM_THREADS = max(1, os.cpu_count() or 1)
GDAL_CACHEMAX_MB = 2048

# 若结果已存在，是否跳过，避免重复耗时计算
SKIP_EXISTING = True

# 是否建立金字塔，方便 GIS 软件显示；会增加一点输出时间
BUILD_OVERVIEWS = True

# =========================================================
# 2. 要生成哪些因子
# =========================================================
RUN_TOPOGRAPHIC_FACTORS = True      # Elevation/Slope/Aspect/Curvature/TPI/TRI/Roughness等
RUN_HYDROLOGY = True                # Whitebox 填洼 + D8 Flow Accumulation + TWI/SPI等
RUN_AP = True                       # AP，没给数据会自动跳过
RUN_CN = True                       # CN，没给数据会自动跳过
RUN_DISTANCE_TO_VECTOR = True       # DTRiver/DTRoad，没给矢量会自动跳过

# 局部地形窗口大小，必须是奇数。
# 30m DEM 下：11≈330m；21≈630m；31≈930m。
LOCAL_WINDOW_SIZE = 11

# 自动提取排水线的汇流累积阈值，单位：像元数。
# 30m DEM 下：500≈0.45km²，1000≈0.90km²，2000≈1.80km²。
DRAINAGE_ACC_THRESHOLD = 1000

# 排水密度窗口大小，必须是奇数。
# 30m DEM 下：31≈930m×930m。
DRAINAGE_DENSITY_WINDOW_SIZE = 31

# =========================================================
# 3. 可选数据：AP / CN / 河流 / 道路 / 排水线
# 没有就保持 None，脚本会跳过相应因子。
# =========================================================

# AP：年降雨量、事件累计降雨量或多年平均降雨量
# 方式 1：已有 AP 栅格，脚本会自动对齐到 DEM
# AP_RASTER_PATH: Optional[Path] = None
AP_RASTER_PATH = Path(r"D:\python\DEM_work\data\Co_work\projected_to_dem_grid\AP_PRISM_clip_by_shp_to_DEM_grid.tif")

# 方式 2：AP 站点 CSV/SHP/GPKG/GeoJSON，用 IDW 插值
AP_POINT_FILE: Optional[Path] = None
# AP_POINT_FILE = Path(r"D:\python\DEM_work\data\rainfall\ap_stations.csv")
AP_FIELD = "AP"           # 降雨字段名
AP_CSV_X = "lon"          # CSV 经度字段
AP_CSV_Y = "lat"          # CSV 纬度字段
AP_CSV_CRS = "EPSG:4326"  # CSV 点坐标系
AP_IDW_K = 8
AP_IDW_POWER = 2.0

# CN：Curve Number
# 方式 1：已有 CN 栅格
# CN_RASTER_PATH: Optional[Path] = None
CN_RASTER_PATH = Path(r"D:\python\DEM_work\data\Co_work\projected_to_dem_grid\CN_GCN250_ARCII_clip_by_shp_to_DEM_grid.tif")

# 方式 2：已有 CN 矢量面，属性表中有 CN 字段
CN_VECTOR_PATH: Optional[Path] = None
# CN_VECTOR_PATH = Path(r"D:\python\DEM_work\data\cn\CN_polygon.shp")
CN_FIELD = "CN"

# 河流、道路、排水线矢量，可选
# RIVER_VECTOR_PATH: Optional[Path] = None
RIVER_VECTOR_PATH = Path(r"D:\python\DEM_work\data\Co_work\projected_vectors\NY_NJ_waterways_to_DEM_CRS.gpkg")

# ROAD_VECTOR_PATH: Optional[Path] = None
ROAD_VECTOR_PATH = Path(r"D:\python\DEM_work\data\Co_work\projected_vectors\NY_NJ_roads_to_DEM_CRS.gpkg")

# 如果已有排水线矢量，可填这里；否则脚本会用 Flow Accumulation 自动提取 DTDrainage
DRAINAGE_VECTOR_PATH: Optional[Path] = None
# DRAINAGE_VECTOR_PATH = Path(r"D:\python\DEM_work\data\vector\drainage.shp")


In [65]:
# =========================================================
# 4. 通用工具函数
# =========================================================

def ensure_odd_window(size: int) -> int:
    size = int(size)
    if size < 3:
        size = 3
    if size % 2 == 0:
        size += 1
    return size


def iter_windows(width: int, height: int, block_size: int):
    for row_off in range(0, height, block_size):
        win_h = min(block_size, height - row_off)
        for col_off in range(0, width, block_size):
            win_w = min(block_size, width - col_off)
            yield Window(col_off, row_off, win_w, win_h)


def count_windows(width: int, height: int, block_size: int) -> int:
    return int(np.ceil(width / block_size) * np.ceil(height / block_size))


def make_profile_like(src, dtype=OUT_DTYPE, nodata=NODATA, count=1):
    profile = src.profile.copy()
    profile.update(
        driver="GTiff",
        count=count,
        dtype=dtype,
        nodata=nodata,
        compress="DEFLATE",
        predictor=3 if dtype in ["float32", "float64"] else 2,
        tiled=True,
        blockxsize=512,
        blockysize=512,
        BIGTIFF="IF_SAFER",
    )
    return profile


def build_overviews(raster_path: Path):
    if not BUILD_OVERVIEWS:
        return
    raster_path = Path(raster_path)
    if not raster_path.exists():
        return
    with rasterio.open(raster_path, "r+") as dst:
        factors = [2, 4, 8, 16, 32, 64]
        dst.build_overviews(factors, Resampling.average)
        dst.update_tags(ns="rio_overview", resampling="average")


def output_exists(path_or_paths) -> bool:
    if not SKIP_EXISTING:
        return False
    if isinstance(path_or_paths, (list, tuple)):
        return all(Path(p).exists() for p in path_or_paths)
    return Path(path_or_paths).exists()


def validate_dem(dem_path: Path) -> Dict:
    if not dem_path.exists():
        raise FileNotFoundError(f"DEM 不存在：{dem_path}")

    with rasterio.open(dem_path) as src:
        info = {
            "path": str(dem_path),
            "crs": src.crs,
            "width": src.width,
            "height": src.height,
            "res": src.res,
            "bounds": src.bounds,
            "nodata": src.nodata,
            "dtype": src.dtypes[0],
        }

        print("\n" + "=" * 80)
        print("[DEM 信息]")
        for k, v in info.items():
            print(f"{k}: {v}")
        print("=" * 80)

        if src.crs is None:
            raise ValueError("DEM 没有 CRS，请先定义坐标系。")
        if src.crs.is_geographic:
            raise ValueError(
                "当前 DEM 是经纬度坐标系。请先投影到米制坐标系，例如 EPSG:5070，"
                "否则坡度、距离、TWI 等结果不可靠。"
            )
    return info


def rasters_match_grid(path1: Path, path2: Path) -> bool:
    with rasterio.open(path1) as a, rasterio.open(path2) as b:
        return (
            a.crs == b.crs
            and a.width == b.width
            and a.height == b.height
            and np.allclose(a.transform, b.transform)
        )


def mask_raster_by_dem_valid(raster_path: Path, dem_path: Path, desc="Mask by DEM"):
    """将 DEM 无效区域对应的输出栅格设置为 NODATA。"""
    with rasterio.open(dem_path) as dem, rasterio.open(raster_path, "r+") as dst:
        src_nodata = dem.nodata if dem.nodata is not None else NODATA
        total = count_windows(dem.width, dem.height, BLOCK_SIZE_TERRAIN)
        for win in tqdm(iter_windows(dem.width, dem.height, BLOCK_SIZE_TERRAIN), total=total, desc=desc, unit="block"):
            dem_block = dem.read(1, window=win, masked=True)
            valid = ~dem_block.mask
            arr = dst.read(1, window=win)
            arr = np.where(valid, arr, NODATA).astype(dst.dtypes[0])
            dst.write(arr, 1, window=win)


def write_blockwise_copy_dem(dem_path: Path, out_path: Path):
    """将 DEM 本身作为 Elevation 因子复制输出。"""
    if output_exists(out_path):
        print(f"[跳过] {out_path.name} 已存在")
        return out_path
    out_path.parent.mkdir(parents=True, exist_ok=True)
    with rasterio.open(dem_path) as src:
        src_nodata = src.nodata if src.nodata is not None else NODATA
        profile = make_profile_like(src, dtype=OUT_DTYPE, nodata=NODATA)
        total = count_windows(src.width, src.height, BLOCK_SIZE_TERRAIN)
        with rasterio.open(out_path, "w", **profile) as dst:
            for win in tqdm(iter_windows(src.width, src.height, BLOCK_SIZE_TERRAIN), total=total, desc="Elevation", unit="block"):
                arr = src.read(1, window=win, masked=True, out_dtype="float32")
                data = np.asarray(arr.filled(NODATA), dtype="float32")
                dst.write(data, 1, window=win)
    build_overviews(out_path)
    return out_path


def clean_geometries(gdf: gpd.GeoDataFrame) -> gpd.GeoDataFrame:
    gdf = gdf[gdf.geometry.notna()].copy()
    gdf = gdf[~gdf.geometry.is_empty].copy()
    # buffer(0) 可修复大部分自相交等问题
    gdf["geometry"] = gdf.geometry.buffer(0)
    gdf = gdf[gdf.geometry.notna()].copy()
    gdf = gdf[~gdf.geometry.is_empty].copy()
    return gdf


def load_vector_to_dem_crs(vector_path: Path, dem_crs) -> gpd.GeoDataFrame:
    vector_path = Path(vector_path)
    if not vector_path.exists():
        raise FileNotFoundError(f"矢量文件不存在：{vector_path}")
    gdf = gpd.read_file(vector_path)
    if gdf.empty:
        raise ValueError(f"矢量为空：{vector_path}")
    if gdf.crs is None:
        raise ValueError(f"矢量没有 CRS，请先定义坐标系：{vector_path}")
    gdf = clean_geometries(gdf)
    gdf = gdf.to_crs(dem_crs)
    return gdf


def align_raster_to_dem(
    src_raster: Path,
    dem_path: Path,
    out_path: Path,
    resampling: Resampling,
    dtype=OUT_DTYPE,
    dst_nodata=NODATA,
):
    """将任意输入栅格重投影/重采样到 DEM 网格。"""
    if output_exists(out_path):
        print(f"[跳过] {out_path.name} 已存在")
        return out_path

    out_path.parent.mkdir(parents=True, exist_ok=True)
    with rasterio.open(dem_path) as dem, rasterio.open(src_raster) as src:
        profile = make_profile_like(dem, dtype=dtype, nodata=dst_nodata)
        print("\n" + "=" * 80)
        print(f"[栅格对齐] 输入: {src_raster}")
        print(f"[栅格对齐] 输出: {out_path}")
        print("=" * 80)
        with rasterio.open(out_path, "w", **profile) as dst:
            reproject(
                source=rasterio.band(src, 1),
                destination=rasterio.band(dst, 1),
                src_transform=src.transform,
                src_crs=src.crs,
                src_nodata=src.nodata,
                dst_transform=dem.transform,
                dst_crs=dem.crs,
                dst_nodata=dst_nodata,
                resampling=resampling,
                num_threads=NUM_THREADS,
                init_dest_nodata=True,
            )
    build_overviews(out_path)
    return out_path


In [66]:
# =========================================================
# 5. DEM 直接派生因子：一次分块读取，输出多个因子
# =========================================================

def local_mean_nan(arr: np.ndarray, size: int) -> np.ndarray:
    valid = np.isfinite(arr)
    arr0 = np.where(valid, arr, 0.0).astype("float32")
    kernel_area = float(size * size)
    local_sum = uniform_filter(arr0, size=size, mode="nearest") * kernel_area
    local_count = uniform_filter(valid.astype("float32"), size=size, mode="nearest") * kernel_area
    mean = np.where(local_count > 0, local_sum / local_count, np.nan)
    return mean.astype("float32")


def local_std_nan(arr: np.ndarray, size: int) -> np.ndarray:
    mean = local_mean_nan(arr, size)
    mean2 = local_mean_nan(arr ** 2, size)
    var = mean2 - mean ** 2
    var = np.where(var > 0, var, 0.0)
    return np.sqrt(var).astype("float32")


def local_min_max_nan(arr: np.ndarray, size: int):
    valid = np.isfinite(arr)
    arr_for_max = np.where(valid, arr, -np.inf).astype("float32")
    arr_for_min = np.where(valid, arr, np.inf).astype("float32")
    local_max = maximum_filter(arr_for_max, size=size, mode="nearest")
    local_min = minimum_filter(arr_for_min, size=size, mode="nearest")
    local_max = np.where(np.isfinite(local_max), local_max, np.nan)
    local_min = np.where(np.isfinite(local_min), local_min, np.nan)
    return local_min.astype("float32"), local_max.astype("float32")


def calculate_tri_3x3(z: np.ndarray) -> np.ndarray:
    center = z[1:-1, 1:-1]
    neighbors = [
        z[:-2, :-2], z[:-2, 1:-1], z[:-2, 2:],
        z[1:-1, :-2],               z[1:-1, 2:],
        z[2:, :-2],  z[2:, 1:-1],  z[2:, 2:],
    ]
    sum_sq = np.zeros_like(center, dtype="float32")
    count = np.zeros_like(center, dtype="float32")
    for nb in neighbors:
        valid = np.isfinite(center) & np.isfinite(nb)
        diff = np.where(valid, nb - center, 0.0).astype("float32")
        sum_sq += diff ** 2
        count += valid.astype("float32")
    tri = np.where(count > 0, np.sqrt(sum_sq), np.nan)
    return tri.astype("float32")


def calculate_topographic_factors(dem_path: Path, out_dir: Path, local_window_size: int = 11) -> List[Path]:
    """
    DEM 直接派生因子，全部分块计算。

    输出：
    - Elevation
    - Slope
    - Aspect
    - Curvature
    - PlanCurvature
    - ProfileCurvature
    - TPI
    - TRI
    - Roughness
    - LocalRelief
    """
    out_dir.mkdir(parents=True, exist_ok=True)
    local_window_size = ensure_odd_window(local_window_size)
    radius = max(local_window_size // 2, 2)

    paths = {
        "Elevation": out_dir / "Elevation.tif",
        "Slope": out_dir / "Slope.tif",
        "Aspect": out_dir / "Aspect.tif",
        "Curvature": out_dir / "Curvature.tif",
        "PlanCurvature": out_dir / "PlanCurvature.tif",
        "ProfileCurvature": out_dir / "ProfileCurvature.tif",
        "TPI": out_dir / "TPI.tif",
        "TRI": out_dir / "TRI.tif",
        "Roughness": out_dir / "Roughness.tif",
        "LocalRelief": out_dir / "LocalRelief.tif",
    }

    if output_exists(list(paths.values())):
        print("[跳过] DEM 直接派生因子均已存在")
        return list(paths.values())

    with rasterio.open(dem_path) as src:
        dx = abs(src.transform.a)
        dy = abs(src.transform.e)
        src_nodata = src.nodata if src.nodata is not None else NODATA
        profile = make_profile_like(src, dtype=OUT_DTYPE, nodata=NODATA)
        total = count_windows(src.width, src.height, BLOCK_SIZE_TERRAIN)

        print("\n" + "=" * 80)
        print("[DEM 直接派生因子] 开始计算")
        print(f"dx={dx:.3f} m, dy={dy:.3f} m")
        print(f"局部窗口: {local_window_size} × {local_window_size}")
        print("=" * 80)

        # 只打开不存在或需要覆盖的输出文件
        datasets = {}
        for name, path in paths.items():
            if path.exists() and SKIP_EXISTING:
                print(f"[跳过单项] {name}: {path.name} 已存在")
                continue
            datasets[name] = rasterio.open(path, "w", **profile)

        try:
            for win in tqdm(iter_windows(src.width, src.height, BLOCK_SIZE_TERRAIN), total=total, desc="Topographic factors", unit="block"):
                h = int(win.height)
                w = int(win.width)
                read_win = Window(
                    win.col_off - radius,
                    win.row_off - radius,
                    win.width + 2 * radius,
                    win.height + 2 * radius,
                )
                z = src.read(1, window=read_win, boundless=True, fill_value=src_nodata, out_dtype="float32")
                z = z.astype("float32")
                valid = np.isfinite(z) & (z != src_nodata)
                z = np.where(valid, z, np.nan)

                center = z[radius:radius + h, radius:radius + w]
                inner_valid = np.isfinite(center)

                # 一阶导数：row方向向南，col方向向东
                dz_dsouth, dz_deast = np.gradient(z, dy, dx, edge_order=1)
                dz_dnorth = -dz_dsouth

                # Slope
                slope_rad = np.arctan(np.sqrt(dz_deast ** 2 + dz_dnorth ** 2))
                slope_deg = np.degrees(slope_rad)
                slope_out = slope_deg[radius:radius + h, radius:radius + w]

                # Aspect：下坡方向，0=北，90=东，180=南，270=西，平坦区=-1
                east_component = -dz_deast
                north_component = -dz_dnorth
                aspect = np.degrees(np.arctan2(east_component, north_component))
                aspect = (aspect + 360.0) % 360.0
                aspect[slope_deg < 1e-6] = -1.0
                aspect_out = aspect[radius:radius + h, radius:radius + w]

                # Curvature：Laplacian 近似
                d2z_dsouth2 = np.gradient(dz_dsouth, dy, axis=0, edge_order=1)
                d2z_deast2 = np.gradient(dz_deast, dx, axis=1, edge_order=1)
                curvature = d2z_deast2 + d2z_dsouth2
                curvature_out = curvature[radius:radius + h, radius:radius + w]

                # Plan/Profile Curvature
                p = dz_deast              # z_x, x=East
                q = -dz_dsouth            # z_y, y=North
                r = np.gradient(p, dx, axis=1, edge_order=1)       # z_xx
                s = -np.gradient(p, dy, axis=0, edge_order=1)      # z_xy
                t = -np.gradient(q, dy, axis=0, edge_order=1)      # z_yy
                grad2 = p ** 2 + q ** 2
                eps = 1e-12
                plan_curv = -(q ** 2 * r - 2 * p * q * s + p ** 2 * t) / (grad2 * np.sqrt(grad2) + eps)
                prof_curv = -(p ** 2 * r + 2 * p * q * s + q ** 2 * t) / (grad2 * (1.0 + grad2) ** 1.5 + eps)
                plan_out = plan_curv[radius:radius + h, radius:radius + w]
                prof_out = prof_curv[radius:radius + h, radius:radius + w]
                grad2_center = grad2[radius:radius + h, radius:radius + w]

                # TPI / Roughness / LocalRelief
                local_mean = local_mean_nan(z, local_window_size)
                local_mean_center = local_mean[radius:radius + h, radius:radius + w]
                tpi_out = center - local_mean_center

                roughness = local_std_nan(z, local_window_size)
                roughness_out = roughness[radius:radius + h, radius:radius + w]

                local_min, local_max = local_min_max_nan(z, local_window_size)
                relief_out = local_max[radius:radius + h, radius:radius + w] - local_min[radius:radius + h, radius:radius + w]

                # TRI 3×3
                tri_full = calculate_tri_3x3(z)
                tri_out = tri_full[radius - 1:radius - 1 + h, radius - 1:radius - 1 + w]

                outputs = {
                    "Elevation": center,
                    "Slope": slope_out,
                    "Aspect": aspect_out,
                    "Curvature": curvature_out,
                    "PlanCurvature": plan_out,
                    "ProfileCurvature": prof_out,
                    "TPI": tpi_out,
                    "TRI": tri_out,
                    "Roughness": roughness_out,
                    "LocalRelief": relief_out,
                }

                for name, arr in outputs.items():
                    if name not in datasets:
                        continue
                    if name in ["PlanCurvature", "ProfileCurvature"]:
                        valid_mask = inner_valid & (grad2_center > eps) & np.isfinite(arr)
                    else:
                        valid_mask = inner_valid & np.isfinite(arr)
                    arr = np.where(valid_mask, arr, NODATA).astype("float32")
                    datasets[name].write(arr, 1, window=win)
        finally:
            for ds in datasets.values():
                ds.close()

    for pth in paths.values():
        if pth.exists():
            build_overviews(pth)

    print("[DEM 直接派生因子] 完成")
    return list(paths.values())


In [67]:
# =========================================================
# 6. AP 与 CN
# =========================================================

def load_points_file(point_file: Path, value_field: str, dem_crs) -> gpd.GeoDataFrame:
    point_file = Path(point_file)
    suffix = point_file.suffix.lower()
    if suffix == ".csv":
        df = pd.read_csv(point_file)
        for col in [AP_CSV_X, AP_CSV_Y, value_field]:
            if col not in df.columns:
                raise ValueError(f"CSV 缺少字段：{col}")
        gdf = gpd.GeoDataFrame(
            df,
            geometry=[Point(xy) for xy in zip(df[AP_CSV_X], df[AP_CSV_Y])],
            crs=AP_CSV_CRS,
        )
    else:
        gdf = gpd.read_file(point_file)
        if gdf.crs is None:
            raise ValueError(f"点文件没有 CRS：{point_file}")
        if value_field not in gdf.columns:
            raise ValueError(f"点文件缺少字段：{value_field}")

    gdf = clean_geometries(gdf)
    gdf = gdf.to_crs(dem_crs)
    gdf[value_field] = gdf[value_field].astype(float)
    gdf = gdf[np.isfinite(gdf[value_field])].copy()
    if gdf.empty:
        raise ValueError("AP 点数据为空或字段值无效。")
    return gdf


def block_xy(transform: Affine, window: Window) -> Tuple[np.ndarray, np.ndarray]:
    rows = np.arange(window.row_off, window.row_off + window.height) + 0.5
    cols = np.arange(window.col_off, window.col_off + window.width) + 0.5
    cc, rr = np.meshgrid(cols, rows)
    xs, ys = transform * (cc, rr)
    return xs.astype("float64"), ys.astype("float64")


def idw_interpolate_points_to_dem(point_file: Path, value_field: str, dem_path: Path, out_path: Path, k: int = 8, power: float = 2.0):
    """将降雨站点 IDW 插值到 DEM 网格，分块进行。"""
    if output_exists(out_path):
        print(f"[跳过] {out_path.name} 已存在")
        return out_path
    out_path.parent.mkdir(parents=True, exist_ok=True)

    with rasterio.open(dem_path) as dem:
        gdf = load_points_file(point_file, value_field, dem.crs)
        coords = np.column_stack([gdf.geometry.x.values, gdf.geometry.y.values])
        values = gdf[value_field].astype("float64").values
        tree = cKDTree(coords)

        profile = make_profile_like(dem, dtype=OUT_DTYPE, nodata=NODATA)
        total = count_windows(dem.width, dem.height, BLOCK_SIZE_IDW)

        print("\n" + "=" * 80)
        print("[AP] IDW 插值")
        print(f"点数量: {len(gdf)}, 字段: {value_field}, k={k}, power={power}")
        print(f"输出: {out_path}")
        print("=" * 80)

        with rasterio.open(out_path, "w", **profile) as dst:
            for win in tqdm(iter_windows(dem.width, dem.height, BLOCK_SIZE_IDW), total=total, desc="AP IDW", unit="block"):
                xs, ys = block_xy(dem.transform, win)
                xy = np.column_stack([xs.ravel(), ys.ravel()])
                kk = min(k, len(values))
                dist, idx = tree.query(xy, k=kk, workers=NUM_THREADS)
                if kk == 1:
                    dist = dist[:, None]
                    idx = idx[:, None]
                eps = 1e-12
                weights = 1.0 / np.maximum(dist, eps) ** power
                pred = np.sum(weights * values[idx], axis=1) / np.sum(weights, axis=1)

                exact = dist < eps
                if exact.any():
                    exact_rows = np.where(exact.any(axis=1))[0]
                    exact_cols = np.argmax(exact[exact_rows], axis=1)
                    pred[exact_rows] = values[idx[exact_rows, exact_cols]]

                pred = pred.reshape((int(win.height), int(win.width))).astype("float32")
                dem_block = dem.read(1, window=win, masked=True)
                pred = np.where(~dem_block.mask, pred, NODATA).astype("float32")
                dst.write(pred, 1, window=win)
    build_overviews(out_path)
    return out_path


def rasterize_attribute_vector(vector_path: Path, dem_path: Path, out_path: Path, field: str, dtype="float32", all_touched=True):
    """将含属性字段的矢量面栅格化到 DEM 网格，例如 CN。"""
    if output_exists(out_path):
        print(f"[跳过] {out_path.name} 已存在")
        return out_path
    out_path.parent.mkdir(parents=True, exist_ok=True)

    with rasterio.open(dem_path) as dem:
        gdf = load_vector_to_dem_crs(vector_path, dem.crs)
        if field not in gdf.columns:
            raise ValueError(f"矢量缺少字段：{field}")
        shapes = []
        for _, row in gdf.iterrows():
            value = row[field]
            if pd.isna(value):
                continue
            shapes.append((row.geometry, float(value)))
        if not shapes:
            raise ValueError("矢量没有有效属性值。")
        print("\n" + "=" * 80)
        print(f"[矢量栅格化] {vector_path} 字段: {field}")
        print(f"输出: {out_path}")
        print("=" * 80)
        arr = rasterize(
            shapes,
            out_shape=(dem.height, dem.width),
            transform=dem.transform,
            fill=NODATA,
            dtype=dtype,
            all_touched=all_touched,
        )
        profile = make_profile_like(dem, dtype=dtype, nodata=NODATA)
        with rasterio.open(out_path, "w", **profile) as dst:
            dst.write(arr.astype(dtype), 1)
    build_overviews(out_path)
    return out_path


def generate_ap_cn_factors(dem_path: Path, out_dir: Path) -> List[Path]:
    generated = []

    # AP
    if RUN_AP:
        ap_out = out_dir / "AP.tif"
        if AP_RASTER_PATH is not None:
            generated.append(align_raster_to_dem(AP_RASTER_PATH, dem_path, ap_out, Resampling.bilinear, dtype="float32"))
        elif AP_POINT_FILE is not None:
            generated.append(idw_interpolate_points_to_dem(AP_POINT_FILE, AP_FIELD, dem_path, ap_out, k=AP_IDW_K, power=AP_IDW_POWER))
        else:
            print("[AP] 未提供 AP_RASTER_PATH 或 AP_POINT_FILE，跳过 AP。")

    # CN
    if RUN_CN:
        cn_out = out_dir / "CN.tif"
        if CN_RASTER_PATH is not None:
            generated.append(align_raster_to_dem(CN_RASTER_PATH, dem_path, cn_out, Resampling.nearest, dtype="float32"))
        elif CN_VECTOR_PATH is not None:
            generated.append(rasterize_attribute_vector(CN_VECTOR_PATH, dem_path, cn_out, field=CN_FIELD, dtype="float32", all_touched=True))
        else:
            print("[CN] 未提供 CN_RASTER_PATH 或 CN_VECTOR_PATH，跳过 CN。")

    return generated


In [68]:
# =========================================================
# 7. 大栅格友好的矢量栅格化与距离计算
# =========================================================

def create_empty_raster_like(dem_path: Path, out_path: Path, dtype="uint8", nodata=0, init_value=0):
    """创建与 DEM 同网格的空栅格。"""
    out_path.parent.mkdir(parents=True, exist_ok=True)
    with rasterio.open(dem_path) as dem:
        profile = make_profile_like(dem, dtype=dtype, nodata=nodata)
        with rasterio.open(out_path, "w", **profile) as dst:
            total = count_windows(dem.width, dem.height, BLOCK_SIZE_TERRAIN)
            block = None
            for win in tqdm(iter_windows(dem.width, dem.height, BLOCK_SIZE_TERRAIN), total=total, desc=f"Init {out_path.name}", unit="block"):
                shape = (int(win.height), int(win.width))
                if block is None or block.shape != shape:
                    block = np.full(shape, init_value, dtype=dtype)
                dst.write(block, 1, window=win)
    return out_path


def rasterize_vector_to_mask_gdal(vector_path: Path, dem_path: Path, out_mask_path: Path, burn_value: int = 1, all_touched: bool = True):
    """
    使用 GDAL 将线/面矢量栅格化为 mask。
    这样比 rasterio.features.rasterize 一次性生成全图数组更省内存。
    """
    if not GDAL_AVAILABLE:
        raise RuntimeError("osgeo.gdal 不可用，无法执行内存友好的 GDAL 栅格化。")
    if output_exists(out_mask_path):
        print(f"[跳过] {out_mask_path.name} 已存在")
        return out_mask_path

    out_mask_path.parent.mkdir(parents=True, exist_ok=True)
    tmp_dir = out_mask_path.parent / "_tmp_vector"
    tmp_dir.mkdir(parents=True, exist_ok=True)
    tmp_gpkg = tmp_dir / f"{out_mask_path.stem}_reprojected.gpkg"
    if tmp_gpkg.exists():
        tmp_gpkg.unlink()

    with rasterio.open(dem_path) as dem:
        dem_crs = dem.crs
        width, height = dem.width, dem.height
        transform = dem.transform
        wkt = dem.crs.to_wkt()

    gdf = load_vector_to_dem_crs(vector_path, dem_crs)
    gdf.to_file(tmp_gpkg, driver="GPKG")

    driver = gdal.GetDriverByName("GTiff")
    ds = driver.Create(str(out_mask_path), width, height, 1, gdal.GDT_Byte, options=[
        "COMPRESS=DEFLATE", "TILED=YES", "BIGTIFF=IF_SAFER"
    ])
    # rasterio Affine: a,b,c,d,e,f；GDAL geotransform: c,a,b,f,d,e
    ds.SetGeoTransform((transform.c, transform.a, transform.b, transform.f, transform.d, transform.e))
    ds.SetProjection(wkt)
    band = ds.GetRasterBand(1)
    band.SetNoDataValue(0)
    band.Fill(0)

    vds = ogr.Open(str(tmp_gpkg))
    layer = vds.GetLayer(0)
    options = [f"BURN_VALUE={burn_value}"]
    if all_touched:
        options.append("ALL_TOUCHED=TRUE")

    print("\n" + "=" * 80)
    print(f"[GDAL 栅格化] {vector_path}")
    print(f"输出 mask: {out_mask_path}")
    print("=" * 80)

    err = gdal.RasterizeLayer(ds, [1], layer, burn_values=[burn_value], options=options)
    if err != 0:
        raise RuntimeError(f"GDAL RasterizeLayer 失败：{vector_path}")

    band.FlushCache()
    ds.FlushCache()
    band = None
    ds = None
    vds = None

    build_overviews(out_mask_path)
    return out_mask_path


def proximity_distance_gdal(feature_mask_path: Path, dem_path: Path, out_dist_path: Path):
    """
    使用 GDAL ComputeProximity 计算到 mask=1 目标的欧氏距离，单位为 DEM 坐标单位，通常是米。
    该方法写盘计算，比 scipy distance_transform_edt 更适合大栅格。
    """
    if not GDAL_AVAILABLE:
        raise RuntimeError("osgeo.gdal 不可用，无法执行 GDAL Proximity 距离计算。")
    if output_exists(out_dist_path):
        print(f"[跳过] {out_dist_path.name} 已存在")
        return out_dist_path

    out_dist_path.parent.mkdir(parents=True, exist_ok=True)

    src_ds = gdal.Open(str(feature_mask_path), gdal.GA_ReadOnly)
    if src_ds is None:
        raise FileNotFoundError(f"无法打开 mask: {feature_mask_path}")

    width = src_ds.RasterXSize
    height = src_ds.RasterYSize
    gt = src_ds.GetGeoTransform()
    proj = src_ds.GetProjection()

    driver = gdal.GetDriverByName("GTiff")
    dst_ds = driver.Create(str(out_dist_path), width, height, 1, gdal.GDT_Float32, options=[
        "COMPRESS=DEFLATE", "PREDICTOR=3", "TILED=YES", "BIGTIFF=IF_SAFER"
    ])
    dst_ds.SetGeoTransform(gt)
    dst_ds.SetProjection(proj)
    dst_band = dst_ds.GetRasterBand(1)
    dst_band.SetNoDataValue(float(NODATA))
    dst_band.Fill(float(NODATA))

    print("\n" + "=" * 80)
    print(f"[GDAL 距离] 输入 mask: {feature_mask_path}")
    print(f"[GDAL 距离] 输出距离: {out_dist_path}")
    print("=" * 80)

    options = ["VALUES=1", "DISTUNITS=GEO", f"NODATA={NODATA}"]
    gdal.ComputeProximity(src_ds.GetRasterBand(1), dst_band, options)

    dst_band.FlushCache()
    dst_ds.FlushCache()
    src_ds = None
    dst_ds = None

    # 将 DEM 无效区设置为 NODATA
    mask_raster_by_dem_valid(out_dist_path, dem_path, desc=f"Mask {out_dist_path.name}")
    build_overviews(out_dist_path)
    return out_dist_path


def distance_to_vector_factor(vector_path: Path, dem_path: Path, out_dir: Path, name: str) -> Path:
    """矢量线/面 -> mask -> 距离栅格。"""
    mask_path = out_dir / f"{name}_mask.tif"
    dist_path = out_dir / f"{name}.tif"
    rasterize_vector_to_mask_gdal(vector_path, dem_path, mask_path, burn_value=1, all_touched=True)
    proximity_distance_gdal(mask_path, dem_path, dist_path)
    return dist_path


In [69]:
def force_assign_crs(raster_path: Path, epsg: int = 5070):
    """
    只修正栅格 CRS 元数据，不重采样、不改像元值、不改 transform。

    用途：
    WhiteboxTools 有时会把 EPSG:5070 写成 LOCAL_CS / EngineeringCRS，
    这会导致 rasterio.reproject 找不到坐标转换路径。
    """
    raster_path = Path(raster_path)

    if not raster_path.exists():
        print(f"[跳过] 文件不存在：{raster_path}")
        return

    with rasterio.open(raster_path, "r+") as ds:
        print("\n" + "=" * 80)
        print(f"[CRS 修复] {raster_path}")
        print("修复前:", ds.crs)

        ds.crs = CRS.from_epsg(epsg)

        print("修复后:", ds.crs)
        print("=" * 80)

In [70]:
from pathlib import Path
import numpy as np
import rasterio
from rasterio.crs import CRS
from rasterio.enums import Resampling

HYDRO_DIR = OUT_DIR / "hydrology"

for p in [
    DEM_PATH,
    HYDRO_DIR / "DEM_wbt_input.tif",
    HYDRO_DIR / "DEM_filled.tif",
    HYDRO_DIR / "FlowAccumulation_cells.tif",
]:
    force_assign_crs(p, epsg=5070)

for p in [
    DEM_PATH,
    HYDRO_DIR / "DEM_filled.tif",
    HYDRO_DIR / "FlowAccumulation_cells.tif",
]:
    with rasterio.open(p) as src:
        print(p.name, src.crs, src.width, src.height, src.transform)


[CRS 修复] D:\python\DEM_work\data\Features_resolution_30_m\DEM.tif
修复前: EPSG:5070
修复后: EPSG:5070

[CRS 修复] D:\python\DEM_work\data\Features_resolution_30_m\hydrology\DEM_wbt_input.tif
修复前: LOCAL_CS["NAD83 / Conus Albers",UNIT["metre",1,AUTHORITY["EPSG","9001"]],AXIS["Easting",EAST],AXIS["Northing",NORTH]]
修复后: EPSG:5070

[CRS 修复] D:\python\DEM_work\data\Features_resolution_30_m\hydrology\DEM_filled.tif
修复前: LOCAL_CS["NAD83 / Conus Albers",UNIT["metre",1,AUTHORITY["EPSG","9001"]],AXIS["Easting",EAST],AXIS["Northing",NORTH]]
修复后: EPSG:5070

[CRS 修复] D:\python\DEM_work\data\Features_resolution_30_m\hydrology\FlowAccumulation_cells.tif
修复前: LOCAL_CS["NAD83 / Conus Albers",UNIT["metre",1,AUTHORITY["EPSG","9001"]],AXIS["Easting",EAST],AXIS["Northing",NORTH]]
修复后: EPSG:5070
DEM.tif EPSG:5070 22189 22912 | 30.00, 0.00, 1324140.00|
| 0.00,-30.00, 2658030.00|
| 0.00, 0.00, 1.00|
DEM_filled.tif EPSG:5070 22189 22912 | 30.00, 0.00, 1324140.00|
| 0.00,-30.00, 2658030.00|
| 0.00, 0.00, 1.00|
FlowAcc

In [71]:
# =========================================================
# 8. Whitebox 水文分析与 DEM 水文因子
# =========================================================
def make_whitebox_compatible_dem(dem_path: Path, out_path: Path):
    """
    生成 WhiteboxTools 兼容 DEM。

    WhiteboxTools 不支持 PREDICTOR=3 的浮点 GeoTIFF。
    因此这里生成一个无 predictor、无压缩的 Float32 GeoTIFF。
    """
    out_path.parent.mkdir(parents=True, exist_ok=True)

    if out_path.exists():
        print(f"[Whitebox DEM] 已存在，跳过转换: {out_path}")
        return out_path

    print("\n" + "=" * 80)
    print("[Whitebox DEM] 开始生成兼容输入 DEM")
    print(f"输入: {dem_path}")
    print(f"输出: {out_path}")
    print("=" * 80)

    with rasterio.open(dem_path) as src:
        profile = src.profile.copy()

        profile.update(
            driver="GTiff",
            dtype="float32",
            count=1,
            tiled=True,
            blockxsize=512,
            blockysize=512,
            BIGTIFF="IF_SAFER",
        )

        # 关键：移除压缩和 predictor
        profile.pop("compress", None)
        profile.pop("predictor", None)

        with rasterio.open(out_path, "w", **profile) as dst:
            for _, window in tqdm(
                list(src.block_windows(1)),
                desc="Create WBT DEM",
                unit="block"
            ):
                arr = src.read(1, window=window, out_dtype="float32")
                dst.write(arr, 1, window=window)

    print("[Whitebox DEM] 完成。")
    return out_path


def run_whitebox_flow_accumulation(dem_path: Path, out_dir: Path):
    """
    使用 WhiteboxTools 进行填洼和 D8 汇流累积。

    返回：
    filled_dem : Path
        填洼后的 DEM
    flow_acc : Path
        D8 汇流累积栅格，单位为 cells
    """
    out_dir.mkdir(parents=True, exist_ok=True)

    wbt_input_dem = out_dir / "DEM_wbt_input.tif"
    filled_dem = out_dir / "DEM_filled.tif"
    flow_acc = out_dir / "FlowAccumulation_cells.tif"

    try:
        import whitebox
    except ImportError:
        print("[水文分析] 没有安装 whitebox，跳过 TWI / FP / DTDrainage 自动提取。")
        print("安装命令：python -m pip install whitebox")
        return None, None

    # 1. 生成 WhiteboxTools 兼容 DEM
    wbt_input_dem = make_whitebox_compatible_dem(
        dem_path=dem_path,
        out_path=wbt_input_dem
    )

    print("\n" + "=" * 80)
    print("[Whitebox] 填洼 + D8 Flow Accumulation")
    print(f"输入 DEM: {wbt_input_dem}")
    print(f"填洼 DEM: {filled_dem}")
    print(f"Flow Accumulation: {flow_acc}")
    print("=" * 80)

    wbt = whitebox.WhiteboxTools()
    wbt.work_dir = str(out_dir)
    wbt.set_verbose_mode(True)

    # 2. 填洼
    if not filled_dem.exists():
        wbt.fill_depressions(
            str(wbt_input_dem),
            str(filled_dem),
            fix_flats=True
        )

        if not filled_dem.exists():
            raise RuntimeError(
                "Whitebox FillDepressions 失败，没有生成 DEM_filled.tif。"
            )
    else:
        print(f"[Whitebox] DEM_filled 已存在，跳过: {filled_dem}")

    # 3. D8 汇流累积
    if not flow_acc.exists():
        wbt.d8_flow_accumulation(
            str(filled_dem),
            str(flow_acc),
            out_type="cells"
        )

        if not flow_acc.exists():
            raise RuntimeError(
                "Whitebox D8FlowAccumulation 失败，没有生成 FlowAccumulation_cells.tif。"
            )
    else:
        print(f"[Whitebox] FlowAccumulation 已存在，跳过: {flow_acc}")

    # 4. 修正 Whitebox 输出 CRS
    force_assign_crs(wbt_input_dem, epsg=5070)
    force_assign_crs(filled_dem, epsg=5070)
    force_assign_crs(flow_acc, epsg=5070)

    build_overviews(flow_acc)

    print("[Whitebox] 水文分析完成。")

    return filled_dem, flow_acc



def rasters_same_grid_ignore_crs(path1: Path, path2: Path, atol=1e-6) -> bool:
    """
    判断两个栅格是否同网格，但暂时忽略 CRS。

    用于处理 Whitebox 输出 CRS 标签异常的情况：
    - width / height 一致
    - transform 一致
    - 分辨率和范围一致
    则说明不需要重投影，只需要修正 CRS 标签。
    """
    with rasterio.open(path1) as a, rasterio.open(path2) as b:
        return (
            a.width == b.width
            and a.height == b.height
            and np.allclose(tuple(a.transform), tuple(b.transform), atol=atol)
        )


def align_flowacc_if_needed(flow_acc_path: Path, dem_path: Path, out_dir: Path) -> Path:
    """
    对 Flow Accumulation 进行网格检查。

    情况 1：
    FlowAccumulation 与 DEM 完全同网格，只是 CRS 标签坏了：
    → 直接把 FlowAccumulation CRS 修成 DEM CRS，不重投影。

    情况 2：
    确实不同网格：
    → 再调用 align_raster_to_dem。
    """
    flow_acc_path = Path(flow_acc_path)
    dem_path = Path(dem_path)

    with rasterio.open(dem_path) as dem:
        dem_crs = dem.crs
        dem_epsg = dem_crs.to_epsg() if dem_crs is not None else 5070

    # 先判断是否只是 CRS 标签问题
    if rasters_same_grid_ignore_crs(flow_acc_path, dem_path):
        print("\n[FlowAcc 对齐检查] FlowAccumulation 与 DEM 已经同网格。")
        print("[FlowAcc 对齐检查] 只修正 CRS 元数据，不做重投影。")

        force_assign_crs(flow_acc_path, epsg=dem_epsg or 5070)
        return flow_acc_path

    # 如果确实不同网格，才真正重投影
    aligned = out_dir / "FlowAccumulation_cells_aligned_to_DEM.tif"

    print("\n[FlowAcc 对齐检查] FlowAccumulation 与 DEM 不同网格，开始重投影/重采样。")

    return align_raster_to_dem(
        flow_acc_path,
        dem_path,
        aligned,
        Resampling.nearest,
        dtype="float32",
        dst_nodata=NODATA
    )


def calculate_hydro_indices(flow_acc_path: Path, slope_path: Path, dem_path: Path, out_dir: Path) -> List[Path]:
    """
    根据 Flow Accumulation 和 Slope 计算：
    - FlowAccumulation_aligned
    - CatchmentArea
    - SCA
    - TWI
    - FP
    - SPI
    - STI
    - LS

    说明：
    FP 在不同文献里含义不完全统一，这里定义为汇流潜势：FP = ln(1 + FlowAccumulation_cells)。
    SPI 使用坡度耦合形式：SPI = ln(1 + SCA * tan(beta))。
    """
    out_dir.mkdir(parents=True, exist_ok=True)
    flow_acc_path = align_flowacc_if_needed(flow_acc_path, dem_path, out_dir)

    paths = {
        "CatchmentArea": out_dir / "CatchmentArea.tif",
        "SCA": out_dir / "SCA.tif",
        "TWI": out_dir / "TWI.tif",
        "FP": out_dir / "FP.tif",
        "SPI": out_dir / "SPI.tif",
        "STI": out_dir / "STI.tif",
        "LS": out_dir / "LS.tif",
    }
    if output_exists(list(paths.values())):
        print("[跳过] 水文指数均已存在")
        return [flow_acc_path] + list(paths.values())

    with rasterio.open(dem_path) as dem, rasterio.open(flow_acc_path) as fac, rasterio.open(slope_path) as slope:
        dx = abs(dem.transform.a)
        dy = abs(dem.transform.e)
        pixel_area = dx * dy
        pixel_size = np.sqrt(pixel_area)
        profile = make_profile_like(dem, dtype=OUT_DTYPE, nodata=NODATA)
        total = count_windows(dem.width, dem.height, BLOCK_SIZE_TERRAIN)

        print("\n" + "=" * 80)
        print("[水文因子] CatchmentArea / SCA / TWI / FP / SPI / STI / LS")
        print(f"pixel_area={pixel_area:.3f} m², pixel_size={pixel_size:.3f} m")
        print("=" * 80)

        datasets = {}
        for name, path in paths.items():
            if path.exists() and SKIP_EXISTING:
                print(f"[跳过单项] {name}: {path.name} 已存在")
                continue
            datasets[name] = rasterio.open(path, "w", **profile)

        try:
            for win in tqdm(iter_windows(dem.width, dem.height, BLOCK_SIZE_TERRAIN), total=total, desc="Hydro indices", unit="block"):
                fac_block = fac.read(1, window=win, masked=True).astype("float32")
                slope_block = slope.read(1, window=win, masked=True).astype("float32")
                dem_block = dem.read(1, window=win, masked=True)
                valid = (~fac_block.mask) & (~slope_block.mask) & (~dem_block.mask)

                fac_cells = np.asarray(fac_block.filled(0), dtype="float32")
                fac_cells = np.maximum(fac_cells, 0.0)

                slope_deg = np.asarray(slope_block.filled(0), dtype="float32")
                slope_rad = np.deg2rad(slope_deg)
                sin_beta = np.maximum(np.sin(slope_rad), 1e-6)
                tan_beta = np.maximum(np.tan(slope_rad), 1e-3)

                catchment_area = (fac_cells + 1.0) * pixel_area
                sca = (fac_cells + 1.0) * pixel_size

                twi = np.log(sca / tan_beta)
                fp = np.log1p(fac_cells)
                spi = np.log1p(sca * tan_beta)
                sti = ((sca / 22.13) ** 0.6) * ((sin_beta / 0.0896) ** 1.3)
                ls = ((sca / 22.13) ** 0.5) * ((sin_beta / 0.0896) ** 1.3)

                arrs = {
                    "CatchmentArea": catchment_area,
                    "SCA": sca,
                    "TWI": twi,
                    "FP": fp,
                    "SPI": spi,
                    "STI": sti,
                    "LS": ls,
                }
                for name, arr in arrs.items():
                    if name not in datasets:
                        continue
                    arr = np.where(valid & np.isfinite(arr), arr, NODATA).astype("float32")
                    datasets[name].write(arr, 1, window=win)
        finally:
            for ds in datasets.values():
                ds.close()

    for p in paths.values():
        if p.exists():
            build_overviews(p)
    return [flow_acc_path] + list(paths.values())


def calculate_fill_depth(filled_dem_path: Path, dem_path: Path, out_dir: Path) -> Path:
    """FillDepth = FilledDEM - OriginalDEM。"""
    out_dir.mkdir(parents=True, exist_ok=True)
    out_path = out_dir / "FillDepth.tif"
    if output_exists(out_path):
        print(f"[跳过] {out_path.name} 已存在")
        return out_path

    if not rasters_match_grid(filled_dem_path, dem_path):
        filled_dem_path = align_raster_to_dem(filled_dem_path, dem_path, out_dir / "DEM_filled_aligned.tif", Resampling.bilinear)

    with rasterio.open(dem_path) as dem, rasterio.open(filled_dem_path) as filled:
        profile = make_profile_like(dem, dtype=OUT_DTYPE, nodata=NODATA)
        total = count_windows(dem.width, dem.height, BLOCK_SIZE_TERRAIN)
        print("\n" + "=" * 80)
        print("[FillDepth] 填洼深度")
        print("=" * 80)
        with rasterio.open(out_path, "w", **profile) as dst:
            for win in tqdm(iter_windows(dem.width, dem.height, BLOCK_SIZE_TERRAIN), total=total, desc="FillDepth", unit="block"):
                dem_block = dem.read(1, window=win, masked=True).astype("float32")
                filled_block = filled.read(1, window=win, masked=True).astype("float32")
                valid = (~dem_block.mask) & (~filled_block.mask)
                depth = np.asarray(filled_block.filled(np.nan)) - np.asarray(dem_block.filled(np.nan))
                depth = np.where(depth > 0, depth, 0.0)
                depth = np.where(valid & np.isfinite(depth), depth, NODATA).astype("float32")
                dst.write(depth, 1, window=win)
    build_overviews(out_path)
    return out_path


def create_drainage_mask_from_flowacc(flow_acc_path: Path, dem_path: Path, out_dir: Path, threshold: float) -> Path:
    """根据 Flow Accumulation 阈值提取排水通道二值图。"""
    out_dir.mkdir(parents=True, exist_ok=True)
    flow_acc_path = align_flowacc_if_needed(flow_acc_path, dem_path, out_dir)
    out_path = out_dir / "Drainage_mask.tif"
    if output_exists(out_path):
        print(f"[跳过] {out_path.name} 已存在")
        return out_path

    with rasterio.open(dem_path) as dem, rasterio.open(flow_acc_path) as fac:
        profile = make_profile_like(dem, dtype="uint8", nodata=0)
        total = count_windows(dem.width, dem.height, BLOCK_SIZE_TERRAIN)
        print("\n" + "=" * 80)
        print("[Drainage_mask] 根据 Flow Accumulation 提取排水线")
        print(f"阈值: {threshold} cells")
        print("=" * 80)
        with rasterio.open(out_path, "w", **profile) as dst:
            for win in tqdm(iter_windows(dem.width, dem.height, BLOCK_SIZE_TERRAIN), total=total, desc="Drainage mask", unit="block"):
                fac_block = fac.read(1, window=win, masked=True).astype("float32")
                dem_block = dem.read(1, window=win, masked=True)
                mask = (~fac_block.mask) & (~dem_block.mask) & (fac_block.filled(0) >= threshold)
                dst.write(mask.astype("uint8"), 1, window=win)
    build_overviews(out_path)
    return out_path


def calculate_drainage_density_from_mask(drainage_mask_path: Path, dem_path: Path, out_dir: Path, window_size: int = 31) -> Path:
    """DrainageDensity = 窗口内水系长度 / 窗口面积，单位 km/km²。"""
    out_dir.mkdir(parents=True, exist_ok=True)
    window_size = ensure_odd_window(window_size)
    radius = window_size // 2
    out_path = out_dir / "DrainageDensity.tif"
    if output_exists(out_path):
        print(f"[跳过] {out_path.name} 已存在")
        return out_path

    with rasterio.open(dem_path) as dem, rasterio.open(drainage_mask_path) as drainage:
        dx = abs(dem.transform.a)
        dy = abs(dem.transform.e)
        pixel_length = np.sqrt(dx * dy)
        pixel_area = dx * dy
        kernel_area = float(window_size * window_size)
        local_area_m2 = kernel_area * pixel_area

        profile = make_profile_like(dem, dtype=OUT_DTYPE, nodata=NODATA)
        total = count_windows(dem.width, dem.height, BLOCK_SIZE_TERRAIN)
        print("\n" + "=" * 80)
        print("[DrainageDensity] 排水密度")
        print(f"窗口: {window_size} × {window_size}, 单位: km/km²")
        print("=" * 80)
        with rasterio.open(out_path, "w", **profile) as dst:
            for win in tqdm(iter_windows(dem.width, dem.height, BLOCK_SIZE_TERRAIN), total=total, desc="Drainage density", unit="block"):
                h, w = int(win.height), int(win.width)
                read_win = Window(win.col_off - radius, win.row_off - radius, win.width + 2 * radius, win.height + 2 * radius)
                drainage_block = drainage.read(1, window=read_win, boundless=True, fill_value=0, out_dtype="uint8")
                drainage_binary = (drainage_block > 0).astype("float32")
                local_stream_pixels = uniform_filter(drainage_binary, size=window_size, mode="nearest") * kernel_area
                local_stream_length_m = local_stream_pixels * pixel_length
                density = (local_stream_length_m / local_area_m2) * 1000.0
                density_center = density[radius:radius + h, radius:radius + w]
                dem_block = dem.read(1, window=win, masked=True)
                density_center = np.where((~dem_block.mask) & np.isfinite(density_center), density_center, NODATA).astype("float32")
                dst.write(density_center, 1, window=win)
    build_overviews(out_path)
    return out_path


In [72]:
# =========================================================
# 9. DTRiver / DTRoad / DTDrainage
# =========================================================

def generate_distance_factors(dem_path: Path, out_dir: Path, drainage_mask_path: Optional[Path] = None) -> List[Path]:
    generated = []

    if not RUN_DISTANCE_TO_VECTOR:
        print("[距离因子] RUN_DISTANCE_TO_VECTOR=False，跳过。")
        return generated

    if not GDAL_AVAILABLE:
        print("[距离因子] GDAL 不可用，跳过 DTRiver/DTRoad/DTDrainage。")
        return generated

    if RIVER_VECTOR_PATH is not None:
        generated.append(distance_to_vector_factor(RIVER_VECTOR_PATH, dem_path, out_dir, "DTRiver"))
    else:
        print("[DTRiver] 未提供 RIVER_VECTOR_PATH，跳过。")

    if ROAD_VECTOR_PATH is not None:
        generated.append(distance_to_vector_factor(ROAD_VECTOR_PATH, dem_path, out_dir, "DTRoad"))
    else:
        print("[DTRoad] 未提供 ROAD_VECTOR_PATH，跳过。")

    # DTDrainage：优先使用排水线矢量；否则使用自动提取的 Drainage_mask
    if DRAINAGE_VECTOR_PATH is not None:
        generated.append(distance_to_vector_factor(DRAINAGE_VECTOR_PATH, dem_path, out_dir, "DTDrainage"))
    elif drainage_mask_path is not None and Path(drainage_mask_path).exists():
        generated.append(proximity_distance_gdal(drainage_mask_path, dem_path, out_dir / "DTDrainage.tif"))
    else:
        print("[DTDrainage] 没有 DRAINAGE_VECTOR_PATH 或 Drainage_mask，跳过。")

    return generated


In [ ]:
# =========================================================
# 10. 主流程
# =========================================================

def main():
    OUT_DIR.mkdir(parents=True, exist_ok=True)

    env_options = {
        "GDAL_NUM_THREADS": "ALL_CPUS",
        "NUM_THREADS": "ALL_CPUS",
        "GDAL_CACHEMAX": GDAL_CACHEMAX_MB,
        "CHECK_WITH_INVERT_PROJ": "YES",
    }

    print("\n" + "#" * 80)
    print("洪水易发性 DEM 因子提取开始")
    print("#" * 80)

    validate_dem(DEM_PATH)

    factor_paths: List[Path] = []
    drainage_mask_path: Optional[Path] = None
    flow_acc_path: Optional[Path] = None
    filled_dem_path: Optional[Path] = None

    with rasterio.Env(**env_options):
        # 1. DEM 直接派生因子
        if RUN_TOPOGRAPHIC_FACTORS:
            factor_paths.extend(calculate_topographic_factors(DEM_PATH, OUT_DIR, LOCAL_WINDOW_SIZE))
        else:
            print("[DEM 直接派生因子] RUN_TOPOGRAPHIC_FACTORS=False，跳过。")

        slope_path = OUT_DIR / "Slope.tif"
        if not slope_path.exists() and RUN_HYDROLOGY:
            raise FileNotFoundError("水文因子需要 Slope.tif，请先运行 RUN_TOPOGRAPHIC_FACTORS=True。")

        # 2. AP / CN
        factor_paths.extend(generate_ap_cn_factors(DEM_PATH, OUT_DIR))

        # 3. 水文分析与水文地形因子
        if RUN_HYDROLOGY:
            hydro_dir = OUT_DIR / "hydrology"
            filled_dem_path, flow_acc_path = run_whitebox_flow_accumulation(DEM_PATH, hydro_dir)

            if flow_acc_path is not None and Path(flow_acc_path).exists():
                factor_paths.extend(calculate_hydro_indices(flow_acc_path, slope_path, DEM_PATH, OUT_DIR))

                if filled_dem_path is not None and Path(filled_dem_path).exists():
                    factor_paths.append(calculate_fill_depth(filled_dem_path, DEM_PATH, OUT_DIR))

                # 自动提取排水线 mask，并计算排水密度
                if DRAINAGE_VECTOR_PATH is None:
                    drainage_mask_path = create_drainage_mask_from_flowacc(flow_acc_path, DEM_PATH, OUT_DIR, DRAINAGE_ACC_THRESHOLD)
                    factor_paths.append(drainage_mask_path)
                    factor_paths.append(calculate_drainage_density_from_mask(drainage_mask_path, DEM_PATH, OUT_DIR, DRAINAGE_DENSITY_WINDOW_SIZE))
        else:
            print("[水文分析] RUN_HYDROLOGY=False，跳过。")

        # 4. 距离因子：DTRiver / DTRoad / DTDrainage
        factor_paths.extend(generate_distance_factors(DEM_PATH, OUT_DIR, drainage_mask_path=drainage_mask_path))

    # 去重并写清单
    factor_paths = [Path(p) for p in factor_paths if p is not None and Path(p).exists()]
    unique_paths = []
    seen = set()
    for p in factor_paths:
        if str(p) not in seen:
            unique_paths.append(p)
            seen.add(str(p))

    manifest = OUT_DIR / "factor_manifest.csv"
    pd.DataFrame({"factor_path": [str(p) for p in unique_paths], "factor_name": [p.stem for p in unique_paths]}).to_csv(manifest, index=False, encoding="utf-8-sig")

    print("\n" + "#" * 80)
    print("洪水易发性 DEM 因子提取完成")
    print("#" * 80)
    print(f"输出目录：{OUT_DIR}")
    print(f"因子清单：{manifest}")
    print("\n已生成 / 已存在因子：")
    for p in unique_paths:
        print(f"  - {p}")

    return unique_paths

# 运行主流程
factor_paths = main()



################################################################################
洪水易发性 DEM 因子提取开始
################################################################################

[DEM 信息]
path: D:\python\DEM_work\data\Features_resolution_30_m\DEM.tif
crs: EPSG:5070
width: 22189
height: 22912
res: (30.0, 30.0)
bounds: BoundingBox(left=1324140.0, bottom=1970670.0, right=1989810.0, top=2658030.0)
nodata: -9999.0
dtype: float32
[跳过] DEM 直接派生因子均已存在
[跳过] AP.tif 已存在
[跳过] CN.tif 已存在
[Whitebox DEM] 已存在，跳过转换: D:\python\DEM_work\data\Features_resolution_30_m\hydrology\DEM_wbt_input.tif

[Whitebox] 填洼 + D8 Flow Accumulation
输入 DEM: D:\python\DEM_work\data\Features_resolution_30_m\hydrology\DEM_wbt_input.tif
填洼 DEM: D:\python\DEM_work\data\Features_resolution_30_m\hydrology\DEM_filled.tif
Flow Accumulation: D:\python\DEM_work\data\Features_resolution_30_m\hydrology\FlowAccumulation_cells.tif
[Whitebox] DEM_filled 已存在，跳过: D:\python\DEM_work\data\Features_resolution_30_m\hydrology\DEM_filled.tif
[W

Hydro indices: 100%|██████████| 132/132 [01:43<00:00,  1.28block/s]



[FillDepth] 填洼深度


FillDepth: 100%|██████████| 132/132 [00:43<00:00,  3.06block/s]



[FlowAcc 对齐检查] FlowAccumulation 与 DEM 已经同网格。
[FlowAcc 对齐检查] 只修正 CRS 元数据，不做重投影。

[CRS 修复] D:\python\DEM_work\data\Features_resolution_30_m\hydrology\FlowAccumulation_cells.tif
修复前: EPSG:5070
修复后: EPSG:5070

[Drainage_mask] 根据 Flow Accumulation 提取排水线
阈值: 1000 cells


...


[DrainageDensity] 排水密度
窗口: 31 × 31, 单位: km/km²


Drainage density: 100%|██████████| 132/132 [00:17<00:00,  7.57block/s]


[跳过] DTRiver_mask.tif 已存在
[跳过] DTRiver.tif 已存在


C:\Users\24658\AppData\Local\Temp\ipykernel_36120\2263019798.py:142: UserWarning: GeoSeries.notna() previously returned False for both missing (None) and empty geometries. Now, it only returns False for missing values. Since the calling GeoSeries contains empty geometries, the result has changed compared to previous versions of GeoPandas.
Given a GeoSeries 's', you can use '~s.is_empty & s.notna()' to get back the old behaviour.

To further ignore this warning, you can do: 
import warnings; warnings.filterwarnings('ignore', 'GeoSeries.notna', UserWarning)
  gdf = gdf[gdf.geometry.notna()].copy()



[GDAL 栅格化] D:\python\DEM_work\data\Co_work\projected_vectors\NY_NJ_roads_to_DEM_CRS.gpkg
输出 mask: D:\python\DEM_work\data\Features_resolution_30_m\DTRoad_mask.tif

[GDAL 距离] 输入 mask: D:\python\DEM_work\data\Features_resolution_30_m\DTRoad_mask.tif
[GDAL 距离] 输出距离: D:\python\DEM_work\data\Features_resolution_30_m\DTRoad.tif



## 运行后检查建议

1. 先在 QGIS/ArcGIS 打开 `Drainage_mask.tif`，检查水系密度。
   - 太密：把 `DRAINAGE_ACC_THRESHOLD` 调大，比如 `2000` 或 `5000`。
   - 太稀：把 `DRAINAGE_ACC_THRESHOLD` 调小，比如 `500`。

2. 做机器学习前，不建议把所有 DEM 因子无脑全放进去。建议先计算相关系数矩阵，剔除高度共线变量，例如 `|r| > 0.85` 或 `|r| > 0.90`。

3. `FP` 在不同文献中的定义不完全一致。本 Notebook 默认：
   - `FP = ln(1 + FlowAccumulation_cells)`，表示汇流潜势；
   - `SPI = ln(1 + SCA * tanβ)`，表示水流动力指数。

4. `DTRiver / DTRoad` 需要你提供河流和道路矢量。没有提供时会跳过。

5. `AP / CN` 不是 DEM 直接能得到的因子，需要外部降雨数据、土地利用/土壤数据或已有 AP/CN 栅格。
